[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wattwatchers/le_completeness_analysis/blob/main/le_completeness_analysis/analysis_colab.ipynb)

# Device id(s) configuration.

There are 2 options to configure devices for analysis.
1. All devices associated with the API key configured in the .env file
2. A list of device ids


In [ ]:
# API key for the Public REST API
# If you want to analyse all devices associated with the API key, set the value of DEVICE_IDS below to an empty list ([])
API_KEY: str = "key_8a4a9f7148124b899734d3ae4efd4c75"

# If you want to analyse a subset of devices, enter the device ids inside the brackets ([]) like this:
# [ "DD1234567890", "DD2345678901"]

DEVICE_IDS: list = []

# User Apps API credentials (used to check device status)
UA_USERNAME: str = "kevin.quigley@wattwatchers.com.au"
UA_PASSWORD: str = "tEKnRqb.EcJeV@8j*cpa"

# Period configuration

Configure the time period to analyse.


In [ ]:
TIMEZONE: str =   "Australia/Sydney"  # The timezone the period is defined in
START_DATE: str = "2025-07-01"        # Period starts at start of START_DATE (date string in the format <YYYY-MM-DD>)
END_DATE: str =   "2026-06-30"        # Period ends at the end of END_DATE (ate string in the format <YYYY-MM-DD>)
                                      # Has to be today or earlier. 
                                      # If today, analysis will be up to the most recent 5 minute boundary at the time the notebook is run.

# Threshold configuration

Configure the data completeness threshold (as a percentage) under which a device is considered problematic

In [ ]:
DATA_COMPLETENESS_THRESHOLD: float = 80

# Other config (you probably won't need to touch these)

In [ ]:
MAX_TPS: int                           = 20             # Maximum # of API requests per second
ENVIRONMENT: str                       = "production"   # API environment (staging or production)
ANALYSE_UNAGGREGATED_DATA: bool        = False          # When False, only data aggregated to daily granularity is generated and analysed
                                                        # When True, data is also kept and analysed at 5m granularity and. Only do this for small sets of devices / short time periods
                                                        # When more than devices MAX_DEVICES_FOR_UNAGGREGATED_DATA are analysed, unaggregated data will not be analysed, regardless of this configuratio
MAX_DEVICES_FOR_UNAGGREGATED_DATA: int  = 200           # Maximum number of devices for which we will allow unaggregated data. If more than this number of devices
                                                        # is analysed, no analysis on unaggregated data is performed, even if ANALYSE_UNAGGREGATED_DATA is set to True

# API client

In [ ]:
import os
import logging
import re

import plotly 
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots

import numpy as np
import pandas as pd
%pip install pendulum
import pendulum
%pip install itables
from itables import show, JavascriptFunction, JavascriptCode

In [ ]:
def get_logger(logging_level: str = "INFO") -> logging.Logger:
    logger = logging.getLogger("notebook")
    logger.setLevel(logging_level)

    # loggers are cached, so if we call this from multiple places we end up with multiple handlers
    if logger.hasHandlers():
        return logger

    stdout_handler = logging.StreamHandler()
    formatter: logging.Formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    stdout_handler.setFormatter(formatter)
    logger.addHandler(stdout_handler)
    return logger

In [ ]:
import time
from functools import partialmethod
from typing import Any

import httpx
from pendulum import DateTime

JSONType = None | bool | int | float | str | list[Any] | dict[str, Any]


class RestError(Exception):
    """
    Error from REST API Client.
    """

    def __init__(
        self,
        message: str,
        request: httpx.Request,
        response: httpx.Response | None = None,
    ):
        super().__init__(message)
        self.request: httpx.Request = request
        self.response: httpx.Response | None = response


class RestAPIClient:

    def __init__(self, base_url: str, requests_per_sec_max: int, **session_kwargs):
        self._base_url = base_url
        self._client = httpx.Client()
        self._requests_per_sec_max = requests_per_sec_max
        self._last_request_time: DateTime | None = None

        for key, value in session_kwargs.items():
            setattr(self._client, key, value)

    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.close()

    def close(self):
        return self._client.close()

    def _throttler(self):
        """
        This method throttles API request based on when the last request was made and the number of maximum number of requests per second configured.
        (the actual frequency of requests can be lower than the maximum allowed if requests take longer to complete than the minimum interval
        between requests)
        """
        if self._last_request_time is None:
            return
        time_since_last_request = self._last_request_time.diff().total_seconds()
        wait_duration = max(0, 1 / self._requests_per_sec_max - time_since_last_request)
        if wait_duration > 0:
            time.sleep(wait_duration)

    def request(self, method: str, path: str, **kwargs) -> tuple[JSONType, RestError]:
        self._throttler()
        try:
            resp = self._client.request(method, f"{self._base_url}/{path}", **kwargs)
            resp.raise_for_status()
            if len(resp.text) == 0:
                return None, None
            return (resp.json(), None)
        except httpx.HTTPStatusError as http_error:
            error_message = http_error.response.json().get("message", "")
            error = RestError(
                f"Error response {http_error.response.status_code} while requesting {http_error.request.url!r}: {error_message}",
                http_error.request,
                http_error.response,
            )
            return (None, error)
        except httpx.RequestError as err:
            error = RestError(
                f"An error occurred while requesting {err.request.url!r}.", err.request
            )
            return (None, error)
        finally:
            self._last_request_time = pendulum.now()

    get = partialmethod(request, "GET")
    post = partialmethod(request, "POST")
    put = partialmethod(request, "PUT")
    patch = partialmethod(request, "PATCH")
    delete = partialmethod(request, "DELETE")
    head = partialmethod(request, "HEAD")
    options = partialmethod(request, "OPTIONS")


In [ ]:
from dataclasses import dataclass
from enum import Enum
import json

@dataclass
class TimeInterval:
    """
    Data class for a time interval
    """

    timestamp_start: int
    timestamp_end: int


class Granularity(Enum):
    """
    Enum for different LE granularities
    """

    FIVE_MINS = "5m"
    FIFTEEN_MINS = "15m"
    THIRTY_MINS = "30m"
    HOUR = "hour"
    DAY = "day"
    WEEK = "week"
    MONTH = "month"


class CallerError(Exception):
    """
    Error by caller of the client.
    """


class PublicApiClient(RestAPIClient):

    def __init__(
        self,
        environment: str,
        api_key: str,
        requests_per_sec_max: int,
        logger: logging.Logger,
    ):
        match environment:
            case "production" | "prod":
                base_url = "https://api-v3.wattwatchers.com.au"
            case "staging":
                base_url = "https://api-v3-stage.wattwatchers.com.au"
            case _:
                # fallback is prod
                base_url = "https://api-v3.wattwatchers.com.au"
        headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        }
        # TODO: use a considered value for number of requests per second
        super().__init__(base_url, requests_per_sec_max, headers=headers)
        self._logger = logger

    def get_devices_list(self) -> tuple[list | None, RestError | None]:
        """
        Retrieves all device ids associated with the API key
        """
        result = super().get("devices")
        return result

    def get_device_status(self, device_id: str) -> tuple[dict | None, RestError | None]:
        """
        Retrieves the status of the device associated with the device_id
        """
        result = super().get(f"devices/{device_id}")
        return result

    def patch_device_status(
        self, device_id: str, payload: dict
    ) -> tuple[dict | None, RestError | None]:
        """
        Patches the device status of the device associated with the device_id
        Used (among other things) to update WiFi credentials
        """
        result = super().patch(f"devices/{device_id}", data=json.dumps(payload))
        return result

    def update_wifi_credentials(
        self, device_id: str, ssid: str | None = None, psk: str | None = None
    ) -> tuple[dict | None, RestError | None]:
        """
        Updates the WiFi credentials of the device associated with the device_id
        If successful, this will cause the device to switch to WiFi comms.
        """
        if ssid is None and psk is None:
            # No credential details provided, return error
            return (
                None,
                CallerError(
                    "Request to update WiFi credentials requires at least one of SSID and PSK to be defined."
                ),
            )

        payload = {"comms": {"wifi": {}}}
        if not (ssid is None):
            payload["comms"]["wifi"]["ssid"] = ssid
        if not (psk is None):
            payload["comms"]["wifi"]["psk"] = psk

        return self.patch_device_status(device_id, payload)

    def reset_wifi_credentials(
        self, device_id: str
    ) -> tuple[dict | None, RestError | None]:
        """
        Resets the WiFi credentials of the device associated with the device_id
        This will cause the device to switch to cellular comms.
        """
        return self.update_wifi_credentials(device_id, "", "")

    def change_switch_state(
        self, device_id: str, switch_id: str, target_state: str
    ) -> tuple[dict | None, RestError | None]:
        """
        Changes the switch state of the switch with id `switch_id` on the device with
        id `device_id` to state `target_state`.
        """
        payload = {
            "id": device_id,
            "switches": [{"id": switch_id, "state": target_state}],
        }
        return self.patch_device_status(device_id, payload)

    def update_se_reporting_interval(
        self, device_id: str, reporting_interval: int
    ) -> tuple[dict | None, RestError | None]:
        """
        Update the SE reporting interval for the device to the requested value
        """
        payload = {"shortEnergyReportingInterval": reporting_interval}
        return super().post(
            f"devices/{device_id}/reporting-interval", data=json.dumps(payload)
        )

    def get_latest_se(
        self, device_id: str, energy_unit: str | None = "kW"
    ) -> tuple[dict | None, RestError | None]:
        if energy_unit is not None and energy_unit in ["kW", "kWh"]:
            params = {"convert[energy]": energy_unit}
            return super().get(f"short-energy/{device_id}/latest", params=params)
        return super().get(f"short-energy/{device_id}/latest")

    def _max_interval_for_granularity(self, granularity: Granularity) -> int:
        """
        Returns the maximum interval for a single energy request based on the granularity
        """
        MAX_INTERVALS_DAYS = {
            Granularity.FIVE_MINS: 7,
            Granularity.FIFTEEN_MINS: 14,
            Granularity.THIRTY_MINS: 31,
            Granularity.HOUR: 90,
            Granularity.DAY: 3 * 365,  # ≈ 3 years
            Granularity.WEEK: 5 * 365,  # ≈ 5 years
            Granularity.MONTH: 10 * 365,  # ≈ 10 yers
        }
        return MAX_INTERVALS_DAYS.get(granularity, 7) * 24 * 3600

    def _calculate_intervals_for(
        self, granularity: Granularity, timestamp_start: int, timestamp_end: int
    ) -> list[tuple[int, int]]:
        """
        Batches an interval based on the maximum interval per request for the given granularity.
        """
        batch_interval = self._max_interval_for_granularity(granularity)
        intervals = [
            TimeInterval(batch_start, min(batch_start + batch_interval, timestamp_end))
            for batch_start in range(timestamp_start, timestamp_end, batch_interval)
        ]
        return intervals

    def _load_energy(
        self,
        endpoint: str,
        device_id: str,
        intervals: list[TimeInterval],
        unit: str = "kWh",
        granularity: Granularity | None = None,
    ) -> tuple[list | None, RestError | None]:

        energy_data = []
        for interval in intervals:
            params = {
                "fromTs": interval.timestamp_start,
                "toTs": interval.timestamp_end,
                "convert[energy]": unit,
            }
            if granularity is not None:
                params["granularity"] = granularity.value

            self._logger.info(
                f"load from {interval.timestamp_start} to {interval.timestamp_end} for {device_id}"
            )
            (result, error) = super().get(endpoint, params=params)
            if error is not None:
                self._logger.error(
                    f"Error retrieving LE data for {device_id} between {interval.timestamp_start} and {interval.timestamp_end}: {error}"
                )
                return (None, error)

            if result is not None:
                energy_data.extend(result)

        return (energy_data, None)

    def load_long_energy(
        self,
        device_id: str,
        timestamp_start: int,
        timestamp_end: int,
        granularity: Granularity = Granularity.FIVE_MINS,
        unit: str = "kWh",
    ) -> tuple[list | None, RestError | None]:

        intervals = self._calculate_intervals_for(
            granularity, timestamp_start, timestamp_end
        )
        return self._load_energy(
            f"long-energy/{device_id}", device_id, intervals, unit, granularity
        )

    def get_first_le(self, device_id: str) -> tuple[list | None, RestError | None]:
        result = super().get(f"long-energy/{device_id}/first")
        return result
    
    def get_latest_le(self, device_id: str) -> tuple[list | None, RestError | None]:
        result = super().get(f"long-energy/{device_id}/latest")
        return result

    def load_short_energy(
        self,
        device_id: str,
        timestamp_start: int,
        timestamp_end: int,
        unit: str = "kWh",
    ) -> tuple[list | None, RestError | None]:

        max_interval = 12 * 3600  # maximum request interval for SE is 12 hours
        intervals = [
            TimeInterval(batch_start, min(batch_start + max_interval, timestamp_end))
            for batch_start in range(timestamp_start, timestamp_end, max_interval)
        ]
        return self._load_energy(
            f"short-energy/{device_id}", device_id, intervals, unit
        )


In [ ]:
logger: logging.Logger = get_logger()
public_api_client = PublicApiClient(ENVIRONMENT, API_KEY, MAX_TPS, logger)

# Data downloading

In [ ]:
def is_valid_device_id(device_id: str, raise_error: bool = False) -> tuple[bool, str]:
  """
  Returns True if the device ID is valid, False if not.
  Simplified version based on lib_common
  """
  DEVICE_ID_PATTERN = "^[B-F]{1}[A-F0-9]{12}$"
  DEVICE_ID_REGEX = re.compile(DEVICE_ID_PATTERN)
  # re.match won't detect trailing space in the device id, but re.fullmatch will.
  if not DEVICE_ID_REGEX.fullmatch(device_id):
      return False
  return True

# Determine devices to analyse
devices: list[str] = []
if DEVICE_IDS is not None and len(DEVICE_IDS) > 0:
  devices = DEVICE_IDS
else:
  # get all devices associated with API key
  result, error = public_api_client.get_devices_list()
  if error is not None:
    logger.error(f'failed to load devices for API key: {error}')
  else:
    devices = result

# filter out any invalid device ids
devices =[d for d in devices if is_valid_device_id(d)]
num_devices = len(devices)
logger.info(f'found {num_devices} devices to analyse')

In [ ]:
time_start = pendulum.parse(START_DATE, tz=TIMEZONE)
time_end = pendulum.parse(END_DATE, tz=TIMEZONE).add(days=1).subtract(seconds=1)

def get_latest_5_min_boundary(dt: DateTime) -> DateTime:
    # Subtract the remainder of minutes to align to the nearest 5-minute boundary
    minutes_to_subtract = dt.minute % 5
    adjusted_time = dt.subtract(minutes=minutes_to_subtract, seconds=dt.second, microseconds=dt.microsecond)
    return adjusted_time

if time_start > time_end:
    raise ValueError(f"Start date ({START_DATE}) has to be earlier than end date ({END_DATE})")
if time_end > pendulum.now(tz=TIMEZONE):
    if time_end > pendulum.now(tz=TIMEZONE).end_of("day"):
        raise ValueError(f"End date ({END_DATE}) needs to be in the past")
    else:
        # Adjust time_end to now (or previous 5m boundary to make life easier on ourselves)
        time_end = get_latest_5_min_boundary(pendulum.now(tz=TIMEZONE))


In [ ]:
UA_API_BASE_URL = "https://ua-api.wattwatchers.net"
ROOT_DIR = os.getcwd()
if not os.path.exists(os.path.join(ROOT_DIR, "pyproject.toml")):
  ROOT_DIR = os.path.abspath(os.path.join(ROOT_DIR, ".."))

PRIMARY_DATA_DIR = os.path.abspath(
  os.path.join(ROOT_DIR, "le_completeness_analysis", "data")
)
NESTED_DATA_DIR = os.path.abspath(
  os.path.join(ROOT_DIR, "le_completeness_analysis", "le_completeness_analysis", "data")
)

DECOMMISSIONED_CSV_PATH = os.path.join(PRIMARY_DATA_DIR, "decomissioned_devices.csv")
DECOMMISSIONED_CSV_MIRROR_PATH = os.path.join(NESTED_DATA_DIR, "decomissioned_devices.csv")
FIRST_HEARD_CSV_PATH = os.path.join(PRIMARY_DATA_DIR, "first_heard_at.csv")
FIRST_HEARD_CSV_MIRROR_PATH = os.path.join(NESTED_DATA_DIR, "first_heard_at.csv")

DECOMMISSION_CHECK_TEST_SAMPLE_SIZE = 5
REQUIRE_UA_STATUS_CHECKS = True
FAIL_ON_DECOMMISSION_STATUS_API_FAILURE = True

def get_ua_token(username: str, password: str) -> str | None:
  if not username or not password:
    logger.warning("UA username/password not set; skipping device status checks.")
    return None
  payload = {"username": username, "password": password}
  try:
    response = httpx.post(f"{UA_API_BASE_URL}/auth", json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()
    token = data.get("token")
    if not token:
      logger.error("UA auth response missing token.")
    return token
  except httpx.HTTPError as err:
    logger.error(f"UA auth failed: {err}")
    return None

def get_device_status(token: str, device_id: str) -> dict | None:
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  try:
    response = httpx.get(f"{UA_API_BASE_URL}/devices/{device_id}", headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()
  except httpx.HTTPError as err:
    logger.error(f"Failed to load UA device status for {device_id}: {err}")
    return None

def _extract_heard_at(status_payload: dict) -> tuple[int | None, int | None]:
  comms = status_payload.get("comms", {}) if status_payload else {}
  cellular = comms.get("cellular", {}) if isinstance(comms, dict) else {}
  first_heard_at = (
    status_payload.get("first_heard_at")
    or status_payload.get("firstHeardAt")
    or comms.get("first_heard_at")
    or comms.get("firstHeardAt")
    or cellular.get("first_heard_at")
    or cellular.get("firstHeardAt")
  )
  last_heard_at = (
    status_payload.get("last_heard_at")
    or status_payload.get("lastHeardAt")
    or comms.get("last_heard_at")
    or comms.get("lastHeardAt")
    or cellular.get("last_heard_at")
    or cellular.get("lastHeardAt")
  )
  return first_heard_at, last_heard_at

def _to_unix_timestamp(value: object) -> int | None:
  if value is None or (isinstance(value, float) and np.isnan(value)):
    return None
  if isinstance(value, (int, np.integer)):
    return int(value)
  if isinstance(value, (float, np.floating)):
    return int(value)
  parsed = pd.to_datetime(value, utc=True, errors="coerce")
  if pd.isna(parsed):
    return None
  return int(parsed.timestamp())

def _extract_retirement_at(status_payload: dict) -> int | None:
  comms = status_payload.get("comms", {}) if status_payload else {}
  if not isinstance(comms, dict):
    comms = {}
  cellular = comms.get("cellular", {}) if isinstance(comms, dict) else {}
  if not isinstance(cellular, dict):
    cellular = {}
  candidate_values = [
    status_payload.get("retired_at"),
    status_payload.get("retiredAt"),
    status_payload.get("decommissioned_at"),
    status_payload.get("decommissionedAt"),
    status_payload.get("retirement_date"),
    status_payload.get("retirementDate"),
    comms.get("retired_at"),
    comms.get("retiredAt"),
    comms.get("decommissioned_at"),
    comms.get("decommissionedAt"),
    cellular.get("retired_at"),
    cellular.get("retiredAt"),
    cellular.get("decommissioned_at"),
    cellular.get("decommissionedAt"),
  ]
  for candidate in candidate_values:
    ts = _to_unix_timestamp(candidate)
    if ts is not None:
      return ts
  return None

def _normalize_status_text(status_payload: dict) -> str:
  comms = status_payload.get("comms", {}) if status_payload else {}
  if not isinstance(comms, dict):
    comms = {}
  cellular = comms.get("cellular", {}) if isinstance(comms, dict) else {}
  if not isinstance(cellular, dict):
    cellular = {}
  status_parts = [
    status_payload.get("service_status"),
    status_payload.get("serviceStatus"),
    status_payload.get("status"),
    status_payload.get("statusLabel"),
    status_payload.get("service_status_label"),
    status_payload.get("serviceStatusLabel"),
    status_payload.get("status_label"),
    comms.get("status"),
    comms.get("statusLabel"),
    # Also check cellular sub-object, consistent with _extract_heard_at
    cellular.get("status"),
    cellular.get("statusLabel"),
    cellular.get("service_status"),
    cellular.get("serviceStatus"),
  ]
  return " ".join([str(value) for value in status_parts if value]).lower()

def _is_decommissioned(status_payload: dict) -> bool:
  status_text = _normalize_status_text(status_payload)
  return any(term in status_text for term in ["decommissioned", "decomissioned", "retired"] )

def _run_decommission_detection_test(token: str, candidate_devices: list[str], sample_size: int = DECOMMISSION_CHECK_TEST_SAMPLE_SIZE) -> dict[str, int]:
  if not candidate_devices:
    return {"tested": 0, "api_successes": 0, "api_failures": 0, "decommissioned_detected": 0}

  tested = 0
  api_successes = 0
  api_failures = 0
  decommissioned_detected = 0

  for device_id in candidate_devices[:sample_size]:
    tested += 1
    payload = get_device_status(token, device_id)
    if payload is None:
      api_failures += 1
      continue
    api_successes += 1
    if _is_decommissioned(payload):
      decommissioned_detected += 1

  return {
    "tested": tested,
    "api_successes": api_successes,
    "api_failures": api_failures,
    "decommissioned_detected": decommissioned_detected,
  }

def _read_first_existing_csv(paths: list[str]) -> pd.DataFrame:
  for path in paths:
    if os.path.exists(path):
      return pd.read_csv(path)
  return pd.DataFrame()

def _save_csv_to_primary_and_mirror(df: pd.DataFrame, primary_path: str, mirror_path: str, label: str) -> None:
  os.makedirs(os.path.dirname(primary_path), exist_ok=True)
  os.makedirs(os.path.dirname(mirror_path), exist_ok=True)
  df.to_csv(primary_path, index=False)
  df.to_csv(mirror_path, index=False)
  logger.info(f"Saved {len(df)} {label} to {primary_path} and {mirror_path}.")

def _load_first_heard_cache() -> dict[str, int | None]:
  df = _read_first_existing_csv([FIRST_HEARD_CSV_PATH, FIRST_HEARD_CSV_MIRROR_PATH])
  if df.empty or "device_id" not in df.columns:
    return {}
  df["first_heard_at"] = pd.to_numeric(df.get("first_heard_at"), errors="coerce")
  return dict(zip(df["device_id"], df["first_heard_at"].where(df["first_heard_at"].notna(), None)))

def _load_decommissioned_cache() -> dict[str, dict[str, int | None]]:
  df = _read_first_existing_csv([DECOMMISSIONED_CSV_PATH, DECOMMISSIONED_CSV_MIRROR_PATH])
  if df.empty or "device_id" not in df.columns:
    return {}
  df["first_heard_at"] = pd.to_numeric(df.get("first_heard_at"), errors="coerce")
  df["last_heard_at"] = pd.to_numeric(df.get("last_heard_at"), errors="coerce")
  if "retirement_at" in df.columns:
    df["retirement_at"] = pd.to_numeric(df.get("retirement_at"), errors="coerce")
  else:
    df["retirement_at"] = df["last_heard_at"]
  return {
    row["device_id"]: {
      "device_id": row["device_id"],
      "first_heard_at": row.get("first_heard_at"),
      "last_heard_at": row.get("last_heard_at"),
      "retirement_at": row.get("retirement_at"),
    }
    for _, row in df.iterrows()
  }

ua_token = get_ua_token(UA_USERNAME, UA_PASSWORD)
if REQUIRE_UA_STATUS_CHECKS and not ua_token:
  raise RuntimeError(
    "Cannot proceed: UA status API token was not obtained, so decommissioned-device checks cannot run."
  )

if ua_token:
  detection_test = _run_decommission_detection_test(ua_token, devices)
  logger.info(
    f"Decommission detection test complete: tested={detection_test['tested']}, "
    f"api_successes={detection_test['api_successes']}, api_failures={detection_test['api_failures']}, "
    f"decommissioned_detected={detection_test['decommissioned_detected']}"
  )
  if detection_test["api_successes"] == 0:
    raise RuntimeError(
      "Cannot proceed: decommission-status API test had zero successful responses."
    )
  if FAIL_ON_DECOMMISSION_STATUS_API_FAILURE and detection_test["api_failures"] > 0:
    raise RuntimeError(
      f"Cannot proceed: decommission-status API test had {detection_test['api_failures']} failed request(s)."
    )

cached_first_heard = _load_first_heard_cache()
cached_decommissioned = _load_decommissioned_cache()
decommissioned_device_info: dict[str, dict[str, int | None]] = dict(cached_decommissioned)
first_heard_info: dict[str, dict[str, int | None]] = {}
status_api_failure_devices: list[str] = []
if ua_token:
  for device_id in devices:
    first_heard_at = cached_first_heard.get(device_id)
    if first_heard_at is None:
      first_le_result, first_le_error = public_api_client.get_first_le(device_id)
      if first_le_error is None and first_le_result is not None:
        first_heard_at = first_le_result.get("timestamp")
      elif first_le_error is not None:
        logger.error(f"Failed to load first LE for device {device_id}: {first_le_error}")

    first_heard_info[device_id] = {
      "device_id": device_id,
      "first_heard_at": first_heard_at,
    }

    if device_id in cached_decommissioned:
      continue
    status_payload = get_device_status(ua_token, device_id)
    if status_payload is None:
      status_api_failure_devices.append(device_id)
      continue
    ua_first_heard_at, ua_last_heard_at = _extract_heard_at(status_payload)
    if first_heard_at is None:
      first_heard_at = ua_first_heard_at
      first_heard_info[device_id]["first_heard_at"] = first_heard_at
    if _is_decommissioned(status_payload):
      retirement_at = _extract_retirement_at(status_payload) or _to_unix_timestamp(ua_last_heard_at)
      logger.info(f"Device {device_id} identified as decommissioned (last heard: {ua_last_heard_at}, retirement: {retirement_at}).")
      decommissioned_device_info[device_id] = {
        "device_id": device_id,
        "first_heard_at": first_heard_at,
        "last_heard_at": ua_last_heard_at,
        "retirement_at": retirement_at,
      }

  if FAIL_ON_DECOMMISSION_STATUS_API_FAILURE and status_api_failure_devices:
    failed_sample = ", ".join(status_api_failure_devices[:10])
    raise RuntimeError(
      f"Cannot proceed: failed to query decommission-status API for {len(status_api_failure_devices)} device(s). "
      f"Sample: {failed_sample}"
    )

  decom_df = pd.DataFrame.from_dict(decommissioned_device_info.values())
  if decom_df.empty:
    decom_df = pd.DataFrame(columns=["device_id", "first_heard_at", "last_heard_at", "retirement_at", "retirement_date"] )
  else:
    decom_df["first_heard_at"] = pd.to_numeric(decom_df.get("first_heard_at"), errors="coerce")
    decom_df["last_heard_at"] = pd.to_numeric(decom_df.get("last_heard_at"), errors="coerce")
    decom_df["retirement_at"] = pd.to_numeric(decom_df.get("retirement_at"), errors="coerce")
    decom_df["retirement_date"] = (
      pd.to_datetime(decom_df["retirement_at"], unit="s", utc=True, errors="coerce")
      .dt.tz_convert(TIMEZONE)
      .dt.date
      .astype("string")
    )
  _save_csv_to_primary_and_mirror(
    decom_df,
    DECOMMISSIONED_CSV_PATH,
    DECOMMISSIONED_CSV_MIRROR_PATH,
    "decommissioned devices"
  )

  if not os.path.exists(DECOMMISSIONED_CSV_PATH):
    logger.error(f"Decommissioned CSV save check failed: file does not exist at {DECOMMISSIONED_CSV_PATH}")
  elif not os.path.exists(DECOMMISSIONED_CSV_MIRROR_PATH):
    logger.error(f"Decommissioned CSV mirror save check failed: file does not exist at {DECOMMISSIONED_CSV_MIRROR_PATH}")
  else:
    try:
      decom_df_saved = pd.read_csv(DECOMMISSIONED_CSV_PATH)
      decom_df_mirror_saved = pd.read_csv(DECOMMISSIONED_CSV_MIRROR_PATH)
      saved_rows = len(decom_df_saved)
      mirror_rows = len(decom_df_mirror_saved)
      saved_unique_devices = decom_df_saved["device_id"].nunique() if "device_id" in decom_df_saved.columns else 0
      expected_rows = len(decom_df)
      if saved_rows != expected_rows or mirror_rows != expected_rows:
        logger.warning(
          f"Decommissioned CSV save check mismatch: expected {expected_rows} rows, found {saved_rows} rows in primary and {mirror_rows} rows in mirror file."
        )
      else:
        logger.info(
          f"Decommissioned CSV save check passed: {saved_rows} rows ({saved_unique_devices} unique devices) in both output paths."
        )
    except Exception as err:
      logger.error(f"Decommissioned CSV save check failed while reading file: {err}")

  first_df = pd.DataFrame.from_dict(first_heard_info.values())
  if first_df.empty:
    first_df = pd.DataFrame(columns=["device_id", "first_heard_at"])
  _save_csv_to_primary_and_mirror(
    first_df,
    FIRST_HEARD_CSV_PATH,
    FIRST_HEARD_CSV_MIRROR_PATH,
    "first_heard_at entries"
  )


In [ ]:
test_device_id = "DD03710135897"
if ua_token:
  test_status_payload = get_device_status(ua_token, test_device_id)
  if test_status_payload:
    print({
      "id": test_status_payload.get("id"),
      "service_status": test_status_payload.get("service_status"),
      "serviceStatus": test_status_payload.get("serviceStatus"),
      "status": test_status_payload.get("status"),
      "statusLabel": test_status_payload.get("statusLabel"),
      "service_status_label": test_status_payload.get("service_status_label"),
      "serviceStatusLabel": test_status_payload.get("serviceStatusLabel"),
      "status_label": test_status_payload.get("status_label"),
    })
    test_comms = test_status_payload.get("comms", {})
    print({
      "comms.status": test_comms.get("status"),
      "comms.statusLabel": test_comms.get("statusLabel"),
      "comms.last_heard_at": test_comms.get("last_heard_at"),
      "comms.lastHeardAt": test_comms.get("lastHeardAt"),
      "comms.first_heard_at": test_comms.get("first_heard_at"),
      "comms.firstHeardAt": test_comms.get("firstHeardAt"),
    })
    print(f"Top-level keys: {sorted(test_status_payload.keys())}")
    print(f"Comms keys: {sorted(test_comms.keys())}")

In [ ]:
# Transform interval data
def process_interval_data(device_id:str, interval_data:list) -> pd.DataFrame:

  def flatten_arrays(item: dict) -> dict:
      """ flatten each element of arrays to their own key. Other types of values are left untouched.
      e.g. {key: [value0, value1, ...]} becomes {key_0: value0, key_1: value1, ...}.            
      """
      flattened = {}
      for key, value in item.items():
        if isinstance(value, list):
          for idx, subvalue in enumerate(value):
            flattened[f"{key}_{idx}"] = subvalue
        else:
          flattened[key] = value
      return flattened

  intervals = []
  for item in interval_data:
      row = flatten_arrays(item)
      intervals.append(row)

  df_intervals = pd.DataFrame.from_dict(intervals) 
  
  df_intervals["device_id"] = device_id
  # Reorder the columns to move 'device_id', 'timestamp', and 'duration' to the front
  columns_order = ['device_id', 'timestamp', 'duration'] + [col for col in df_intervals.columns if col not in ['device_id', 'timestamp', 'duration']]
  df_intervals = df_intervals[columns_order]

  return df_intervals

In [ ]:
# By-day analysis
def roll_up_to_daily(df_intervals: pd.DataFrame) -> pd.DataFrame:
  df_intervals['datetime_end'] = pd.to_datetime(df_intervals['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert(TIMEZONE)
  df_intervals['datetime_start'] = pd.to_datetime(df_intervals['timestamp'] - 300, unit='s').dt.tz_localize('UTC').dt.tz_convert(TIMEZONE)

  # Aggregate based on `datetime_start` to ensure intervals are attributed to the correct day
  df_daily_counts = df_intervals.groupby(['device_id', df_intervals['datetime_start'].dt.date]).size().reset_index(name='entry_count')
  df_daily_counts.columns = ['device_id', 'date', 'num_intervals']
  # Keep dates as naive datetime.date objects so they can be safely merged with
  # cached data (which also uses naive dates). add_completeness handles tz-localisation.
  df_daily_counts['date'] = pd.to_datetime(df_daily_counts['date']).dt.date

  return df_daily_counts


In [ ]:
def add_missing_days(df: pd.DataFrame | None, device_id: str, time_start: DateTime, time_end: DateTime) -> pd.DataFrame:
  interval = pendulum.interval(time_start, time_end)
  date_range = interval.range('days')
  missing_entries = set([d.date() for d in date_range])
  if df is not None and not df.empty and "date" in df.columns:
    le_dates = set(df['date'].unique())
    missing_entries = missing_entries.difference(le_dates)
  missing_df = pd.DataFrame(list(missing_entries), columns=['date'])
  if missing_df.empty:
    return df
  
  missing_df['device_id'] = device_id
  missing_df['num_intervals'] = 0
  df = pd.concat([df, missing_df], ignore_index=True)
  return df


def add_completeness(
  df: pd.DataFrame,
  device_is_initialised: bool,
  timestamp_device_first_le: int | None,
  timestamp_device_first_heard: int | None,
  timestamp_analysis_end: int,
  ) -> pd.DataFrame:

  df = df.copy()
  # Correctly localize naive date objects (from cache) as TIMEZONE midnight, not UTC midnight.
  # Using utc=True would shift day boundaries by the UTC offset (e.g. 10h for AEST),
  # causing timestamp_end to wrongly exceed timestamp_analysis_end and under-estimate expected intervals.
  _dates = pd.to_datetime(df['date'])
  if _dates.dt.tz is None:
    df['date'] = _dates.dt.tz_localize(TIMEZONE, ambiguous='infer', nonexistent='shift_forward')
  else:
    df['date'] = _dates.dt.tz_convert(TIMEZONE)

  def _num_minutes(date: pd.Timestamp) -> int:
    start_date = pendulum.instance(date).start_of('day')
    end_date = start_date.add(days=1)
    minutes_in_day = end_date.diff(start_date).in_minutes()
    return minutes_in_day
  
  df['num_minutes'] = df['date'].apply(_num_minutes)

  if not device_is_initialised:
    df['num_intervals_expected'] = 0
  else:
    df['num_intervals_expected'] = df['num_minutes'] // 5

  # temporarily add timestamp columns based on date
  df['timestamp_start'] = df['date'].apply(lambda x: int(x.timestamp()))
  df['timestamp_end'] = df['timestamp_start'] + df['num_minutes'] * 60
  # Adjust timestamp_start for end-of-interval timestamps
  df['timestamp_start'] = df['timestamp_start'] - 300

  if device_is_initialised:

    effective_first_ts = timestamp_device_first_le
    if timestamp_device_first_heard is not None:
      if effective_first_ts is None:
        effective_first_ts = timestamp_device_first_heard
      else:
        effective_first_ts = max(effective_first_ts, timestamp_device_first_heard)

    # ADJUST EXPECTED NUMBER OF INTERVALS BASED ON FIRST_LE/FIRST_HEARD FOR DEVICE AND THE END OF ANALYSIS TIMESTAMP
    # This only needs to happen for initialised devices. Uninitialised devices by definition have 0 expected intervals (which is handled above)

    # HANDLE FULL DAYS BEFORE DEVICE WAS INITIALISED/HEARD
    # when timestamp_end < day of effective_first_ts, num_intervals_expected = 0
    if effective_first_ts is not None:
      df.loc[df['timestamp_end'] < effective_first_ts, 'num_intervals_expected'] = 0

    # HANDLE DAY AT WHICH DEVICE WAS INITIALISED/HEARD
    # (i.e. we expect some but not all intervals in this case)

    # when effective_first_ts is between timestamp_start and timestamp_end,
    # num_intervals_expected depends on effective_first_ts (i.e. num intervals between effective_first_ts and end of day)
    if effective_first_ts is not None:
      mask = (df['timestamp_start'] <= effective_first_ts) & (df['timestamp_end'] > effective_first_ts)
      # The +1 is there because if effective_first_ts is equal to timestamp_end, that means there is 1 expected interval (the one represented by effective_first_ts)
      df.loc[mask, 'num_intervals_expected'] = ((df.loc[mask, 'timestamp_end'] - effective_first_ts) // 300).astype(int) + 1

    # HANDLE PARTIAL DAY AT END OF INTERVAL
    # i.e. when last day of analysis is current day (and we only expect intervals up to current time)
    mask = (df['timestamp_end'] > timestamp_analysis_end)
    # For partial day at end of interval, follow same logic as for partial days at start of interval
    # (but subtract the number of intervals between end of day and timestamp_end from the expected number of intervals for a full day)
    df.loc[mask, 'num_intervals_expected'] = df.loc[mask, 'num_intervals_expected'] - (((df.loc[mask, 'timestamp_end'] - timestamp_analysis_end) // 300).astype(int) + 1)
    
  df['missing_intervals'] = df['num_intervals_expected'] - df['num_intervals']
  df['interval_completeness'] = df.apply(
      lambda row: 100 if row['num_intervals_expected'] == 0 else 100 * row['num_intervals'] / row['num_intervals_expected'], 
      axis=1
  )
  df['date'] = df['date'].dt.date
  return df

def move_column_to_index(df: pd.DataFrame, column_name: str, index: int) -> pd.DataFrame:
  cols = df.columns.tolist()
  # TODO: add check for column name and num columns
  cols.insert(index, cols.pop(cols.index(column_name)))
  df = df[cols]
  return df

DATA_DIR = os.path.abspath(
  os.path.join((ROOT_DIR if 'ROOT_DIR' in globals() else os.getcwd()), "le_completeness_analysis", "data")
)

def _device_csv_path(device_id: str) -> str:
  return os.path.join(DATA_DIR, f"{device_id}.csv")

def _read_device_daily_cache(device_id: str) -> pd.DataFrame:
  path = _device_csv_path(device_id)
  if not os.path.exists(path):
    return pd.DataFrame()
  df = pd.read_csv(path)
  if df.empty:
    return df
  df = df.rename(columns={
    "interval_compleness": "interval_completeness",
    "num_intervals_num_minutes": "num_minutes",
    "num_invervals_expected": "num_intervals_expected",
    "missing_invervals": "missing_intervals",
  })
  if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date
  if "num_intervals" not in df.columns:
    if "num_intervals_expected" in df.columns and "missing_intervals" in df.columns:
      df["num_intervals"] = df["num_intervals_expected"] - df["missing_intervals"]
    else:
      df["num_intervals"] = 0
  return df

def _merge_device_daily(existing: pd.DataFrame, new: pd.DataFrame | None) -> pd.DataFrame:
  if new is None or new.empty:
    return existing
  if existing is None or existing.empty:
    return new
  combined = pd.concat([existing, new], ignore_index=True)
  combined = combined.drop_duplicates(subset=["device_id", "date"], keep="last")
  return combined

def _all_dates_in_range(time_start: DateTime, time_end: DateTime) -> list:
  interval = pendulum.interval(time_start, time_end)
  return [d.date() for d in interval.range('days')]

def _missing_date_ranges(missing_dates: set) -> list[tuple]:
  if not missing_dates:
    return []
  sorted_dates = sorted(missing_dates)
  ranges = []
  range_start = sorted_dates[0]
  prev = sorted_dates[0]
  for current in sorted_dates[1:]:
    if (current - prev).days == 1:
      prev = current
      continue
    ranges.append((range_start, prev))
    range_start = current
    prev = current
  ranges.append((range_start, prev))
  return ranges

def _date_range_to_ts(start_date, end_date) -> tuple[int, int]:
  start_dt = pendulum.datetime(start_date.year, start_date.month, start_date.day, tz=TIMEZONE).start_of("day").add(minutes=5)
  end_dt = pendulum.datetime(end_date.year, end_date.month, end_date.day, tz=TIMEZONE).end_of("day").add(minutes=5)
  return start_dt.int_timestamp, end_dt.int_timestamp

def save_device_daily_completeness(df_device_daily: pd.DataFrame, device_id: str) -> None:
  if df_device_daily is None or df_device_daily.empty:
    return
  os.makedirs(DATA_DIR, exist_ok=True)
  export_df = df_device_daily.copy()
  export_df["date"] = pd.to_datetime(export_df["date"], errors="coerce").dt.date
  export_df = export_df[[
    "device_id",
    "date",
    "interval_completeness",
    "num_minutes",
    "num_intervals_expected",
    "timestamp_start",
    "timestamp_end",
    "missing_intervals",
  ]]
  export_df = export_df.rename(columns={
    "interval_completeness": "interval_compleness",
    "num_minutes": "num_intervals_num_minutes",
    "num_intervals_expected": "num_invervals_expected",
    "missing_intervals": "missing_invervals",
  })
  path = _device_csv_path(device_id)
  if os.path.exists(path):
    existing = pd.read_csv(path)
    if "date" in existing.columns:
      existing["date"] = pd.to_datetime(existing["date"], errors="coerce").dt.date
    export_df = pd.concat([existing, export_df], ignore_index=True)
    export_df = export_df.drop_duplicates(subset=["device_id", "date"], keep="last")
  export_df = export_df.sort_values(by="date", key=lambda s: pd.to_datetime(s, errors="coerce"))
  export_df.to_csv(path, index=False)


In [ ]:
# Download LE data
LE_REQUEST_MAX_RETRIES = 2
LE_REQUEST_RETRY_BACKOFF_SECONDS = 2

def get_le_request_interval(first_le_timestamp: int, latest_le_timestamp: int, timestamp_start: int, timestamp_end: int) -> tuple[int|None, int|None]:
  # If latest LE is older than timestamp start, there will not be any LE data for the whole analysis period
  if latest_le_timestamp < timestamp_start:
    return None, None
  # If first LE is newer than timestamp end, there will not be any LE data for the whole analysis period
  elif first_le_timestamp > timestamp_end:
    return None, None
  else:
    timestamp_start_adjusted = first_le_timestamp if first_le_timestamp > timestamp_start else timestamp_start
    timestamp_end_adjusted = latest_le_timestamp if latest_le_timestamp < timestamp_end else timestamp_end
    return timestamp_start_adjusted, timestamp_end_adjusted

def _load_long_energy_with_retry(
  device_id: str,
  request_timestamp_start: int,
  request_timestamp_end: int,
  max_retries: int = LE_REQUEST_MAX_RETRIES,
  backoff_seconds: int = LE_REQUEST_RETRY_BACKOFF_SECONDS,
  ) -> tuple[list | None, Exception | None]:
  last_error = None
  for attempt in range(max_retries + 1):
    result, error = public_api_client.load_long_energy(
      device_id, request_timestamp_start, request_timestamp_end
    )
    if error is None:
      if attempt > 0:
        logger.info(
          f"Recovered LE request for {device_id} ({request_timestamp_start} -> {request_timestamp_end}) on retry {attempt}/{max_retries}."
        )
      return result, None

    last_error = error
    if attempt < max_retries:
      sleep_seconds = backoff_seconds * (2 ** attempt)
      logger.warning(
        f"Retrying LE request for {device_id} ({request_timestamp_start} -> {request_timestamp_end}) "
        f"after error ({attempt + 1}/{max_retries + 1} attempts): {error}. Waiting {sleep_seconds}s."
      )
      time.sleep(sleep_seconds)

  return None, last_error

def device_is_initialised(first_le_timestamp: int | None) -> bool:
  return first_le_timestamp is not None

def first_le(device_id: str) ->  int | None:
  """
  Returns the timestamp of the first LE for a device.
  Returns None if device not initialised in the requested period.
  Returns 0 if request for first LE fails (this will result in request for LE data will not taking first LE into account)
  """
  result, error = public_api_client.get_first_le(device_id)
  if error is not None:
    logger.error(f'Failed to load first LE for device: {device_id}: {error}')
    return 0
  else:
    if result is None:
      return None
    return result.get('timestamp', 0)

def latest_le(device_id: str) ->  int | None:
  """
  Returns the timestamp of the latest LE for a device.
  Returns None if device not initialised
  Returns current timestamp if request for latest LE fails (this will result in request for LE data not taking latest LE into account)
  """
  result, error = public_api_client.get_latest_le(device_id)
  timestamp_now = pendulum.now(tz=TIMEZONE).int_timestamp
  if error is not None:
    logger.error(f'Failed to load latest LE for device: {device_id}: {error}')
    return timestamp_now
  else:
    if result is None:
      return None
    return result.get('timestamp', timestamp_now)
  

def _to_int_timestamp(value: object) -> int | None:
  if value is None or (isinstance(value, float) and np.isnan(value)):
    return None
  if isinstance(value, (int, np.integer)):
    return int(value)
  if isinstance(value, (float, np.floating)):
    return int(value)
  parsed = pd.to_datetime(value, utc=True, errors='coerce')
  if pd.isna(parsed):
    return None
  return int(parsed.timestamp())

def _device_analysis_end(device_id: str, default_end: DateTime) -> DateTime:
  device_info = decommissioned_device_info.get(device_id) if 'decommissioned_device_info' in globals() else None
  if device_info:
    retirement_at = _to_int_timestamp(device_info.get('retirement_at'))
    last_heard_at = _to_int_timestamp(device_info.get('last_heard_at'))
    # Prefer explicit retirement timestamp; fallback to last heard if retirement is not available.
    cutoff_timestamp = retirement_at if retirement_at is not None else last_heard_at
    if cutoff_timestamp is not None:
      cutoff_dt = pendulum.from_timestamp(cutoff_timestamp, tz=TIMEZONE).end_of('day')
      if cutoff_dt < default_end:
        return cutoff_dt
  return default_end

def _device_first_heard(device_id: str) -> int | None:
  device_info = first_heard_info.get(device_id) if 'first_heard_info' in globals() else None
  if device_info:
    return device_info.get("first_heard_at")
  return None

# Add 5 minutes to time_start and time_end to adjust for fact that interval are *up to* timestamp
timestamp_start = time_start.add(minutes=5).int_timestamp

df_unaggregated: pd.DataFrame = pd.DataFrame()
df_daily_counts: pd.DataFrame = pd.DataFrame()
for index, device_id in enumerate(devices):
  logger.info(f'Downloading LE data for device {index+1}/{num_devices} - {device_id}')

  device_time_end = _device_analysis_end(device_id, time_end)
  if device_time_end < time_start:
    logger.info(f"Skipping {device_id} because retirement/decommission date is before analysis window.")
    continue
  device_timestamp_end = device_time_end.add(minutes=5).int_timestamp
  all_dates = _all_dates_in_range(time_start, device_time_end)

  df_device_cached = _read_device_daily_cache(device_id)
  cached_dates = set(df_device_cached['date'].dropna()) if not df_device_cached.empty else set()
  missing_dates = set(all_dates) - cached_dates

  first_le_timestamp = first_le(device_id)
  latest_le_timestamp = latest_le(device_id)
  first_heard_timestamp = _device_first_heard(device_id)
  is_initialised = device_is_initialised(first_le_timestamp)

  df_device_daily_new = None
  le_request_failed = False
  if is_initialised and missing_dates:
    missing_ranges = _missing_date_ranges(missing_dates)
    result_all = []
    for start_date, end_date in missing_ranges:
      range_start, range_end = _date_range_to_ts(start_date, end_date)
      request_timestamp_start, request_timestamp_end = get_le_request_interval(
        first_le_timestamp, latest_le_timestamp, range_start, range_end
      )

      if request_timestamp_start is None:
        continue

      result, error = _load_long_energy_with_retry(
        device_id, request_timestamp_start, request_timestamp_end
      )
      if error is not None:
        le_request_failed = True
        logger.error(f'Failed to load LE for device {device_id} between {request_timestamp_start} and {request_timestamp_end}: {error}')
      else:
        if result is not None and len(result) == 0:
          logger.info(f"No LE data found for {device_id}")
      if result is not None:
        result_all.extend(result)

    if len(result_all) > 0:
      df_device_intervals = process_interval_data(device_id, result_all)

      # Process into days, include # intervals, # expected, # missing
      df_device_daily_new = roll_up_to_daily(df_device_intervals)

      # Keep unaggregated interval data if requested and we're not analysing too many devices (as that would cause memory issues)
      if ANALYSE_UNAGGREGATED_DATA and len(devices) <= MAX_DEVICES_FOR_UNAGGREGATED_DATA:
        # Only keep columns of interest
        df_device_intervals = df_device_intervals[["device_id", "timestamp", "duration"]]
        df_unaggregated = df_device_intervals if df_unaggregated.empty else pd.concat([df_unaggregated, df_device_intervals])

    # Data hygiene pass: after first pass, explicitly retry only the dates still missing from cache + fresh download.
    downloaded_dates = set(df_device_daily_new['date'].dropna()) if df_device_daily_new is not None and not df_device_daily_new.empty else set()
    missing_dates_after_first_pass = (set(all_dates) - cached_dates) - downloaded_dates
    if missing_dates_after_first_pass:
      logger.info(
        f"Data hygiene for {device_id}: backfilling {len(missing_dates_after_first_pass)} still-missing date(s)."
      )
      hygiene_result_all = []
      for start_date, end_date in _missing_date_ranges(missing_dates_after_first_pass):
        range_start, range_end = _date_range_to_ts(start_date, end_date)
        request_timestamp_start, request_timestamp_end = get_le_request_interval(
          first_le_timestamp, latest_le_timestamp, range_start, range_end
        )
        if request_timestamp_start is None:
          continue
        result, error = _load_long_energy_with_retry(
          device_id, request_timestamp_start, request_timestamp_end
        )
        if error is not None:
          le_request_failed = True
          logger.error(
            f"Data hygiene backfill failed for {device_id} between {request_timestamp_start} and {request_timestamp_end}: {error}"
          )
        elif result is not None:
          hygiene_result_all.extend(result)

      if hygiene_result_all:
        df_hygiene_daily = roll_up_to_daily(process_interval_data(device_id, hygiene_result_all))
        df_device_daily_new = _merge_device_daily(df_device_daily_new, df_hygiene_daily)

  if le_request_failed and (df_device_cached.empty and (df_device_daily_new is None or df_device_daily_new.empty)):
    logger.warning(f"Skipping {device_id} because LE requests failed and no cache/new data is available.")
    continue
  if le_request_failed:
    logger.warning(f"Continuing {device_id} with partial data (cache + successful ranges).")

  df_device_daily = _merge_device_daily(df_device_cached, df_device_daily_new)
  if not df_device_daily.empty:
    df_device_daily["date"] = pd.to_datetime(df_device_daily["date"], errors="coerce").dt.date
    start_date = time_start.date()
    end_date = device_time_end.date()
    df_device_daily = df_device_daily[(df_device_daily["date"] >= start_date) & (df_device_daily["date"] <= end_date)].copy()
  df_device_daily = add_missing_days(df_device_daily, device_id, time_start, device_time_end)
  df_device_daily = add_completeness(
    df_device_daily,
    is_initialised,
    first_le_timestamp,
    first_heard_timestamp,
    device_timestamp_end,
  ) # TOOD: do we need to subtract 5 minutes from request_timestamp_start?

  # Check for overestimated completeness (>100%) — indicates a calculation error.
  # Flag and exclude affected rows from the output.
  overestimated_mask = df_device_daily['interval_completeness'] > 100
  if overestimated_mask.any():
    for _, row in df_device_daily[overestimated_mask].iterrows():
      logger.warning(
        f"[OVERESTIMATED] Device {device_id} on {row.get('date', '?')}: "
        f"{row.get('interval_completeness', 0):.2f}% completeness "
        f"(received={int(row.get('num_intervals', 0))}, "
        f"expected={int(row.get('num_intervals_expected', 0))}). Excluded from output."
      )
    df_device_daily = df_device_daily[~overestimated_mask].copy()

  save_device_daily_completeness(df_device_daily, device_id)
  df_daily_counts = df_device_daily if df_daily_counts.empty else pd.concat([df_daily_counts, df_device_daily])

logger.info(f'Successfully downloaded LE data for {len(df_daily_counts["device_id"].unique())}/{num_devices} devices')

df_daily_counts = df_daily_counts.reset_index(drop=True)

# Re-order columns
df_daily_counts = move_column_to_index(df_daily_counts, 'interval_completeness', 2)

In [ ]:
# Diagnostics: devices with zero intervals
df_device_summary = df_daily_counts.groupby("device_id").agg(
    num_intervals_sum=("num_intervals", "sum"),
    num_expected_sum=("num_intervals_expected", "sum"),
    days_with_data=("num_intervals", lambda s: int((s > 0).sum())),
    days_total=("num_intervals", "size"),
).reset_index()

df_zero_devices = df_device_summary[df_device_summary["num_intervals_sum"] == 0]
if df_zero_devices.empty:
    print("No devices with zero intervals.")
else:
    print("Devices with zero intervals:")
    print(df_zero_devices.sort_values("device_id").to_string(index=False))

In [ ]:
# Check whether df_daily is empty
if df_daily_counts.empty:
    raise ValueError("No LE data was downloaded for any device.")

# Analysis


In [ ]:

df_aggregated = df_daily_counts.groupby('device_id').agg({
    'num_intervals': 'sum',
    'num_intervals_expected': 'sum'
}).reset_index()

devices_with_zero_intervals = df_aggregated[df_aggregated['num_intervals'] == 0]['device_id'].tolist()


df_daily_counts['date'] = pd.to_datetime(df_daily_counts['date'])
df_daily_counts = df_daily_counts.set_index('date')

# Aggregate by device by month
df_monthly_counts = df_daily_counts.groupby(['device_id', pd.Grouper(freq='ME')]).agg({
    'num_intervals': 'sum',
    'num_intervals_expected': 'sum'
}).reset_index()
df_monthly_counts['date'] = df_monthly_counts['date'].dt.to_period('M').astype(str)

# Aggregate across all devices by day
df_daily_counts_all = df_daily_counts.groupby('date').agg({
    'num_intervals': 'sum',
    'num_intervals_expected': 'sum'
}).reset_index()

# Aggregate across all devices by month
df_daily_counts_all['date'] = pd.to_datetime(df_daily_counts_all['date'])
df_daily_counts_all = df_daily_counts_all.set_index('date')
df_monthly_counts_all = df_daily_counts_all.groupby(pd.Grouper(freq='ME')).agg({
    'num_intervals': 'sum',
    'num_intervals_expected': 'sum'
}).reset_index()
df_monthly_counts_all['date'] = df_monthly_counts_all['date'].dt.to_period('M').astype(str)

def add_completeness_columns(df: pd.DataFrame) -> pd.DataFrame:
    df['num_intervals_missing'] = df['num_intervals_expected'] - df['num_intervals']
    df['interval_completeness'] = np.where(df['num_intervals_expected'] > 0,
                                         100 * df['num_intervals'] / df['num_intervals_expected'],
                                         100)
    # df = df.drop(columns=['num_intervals_expected'])
    if 'date' in df.columns:
        df = df.sort_values(by='date')
    return df

df_aggregated = add_completeness_columns(df_aggregated)

df_monthly_counts = add_completeness_columns(df_monthly_counts)
df_monthly_counts = move_column_to_index(df_monthly_counts, 'interval_completeness', 2)

df_daily_counts_all = add_completeness_columns(df_daily_counts_all)
df_daily_counts_all = df_daily_counts_all.reset_index()
df_daily_counts_all = move_column_to_index(df_daily_counts_all, 'interval_completeness', 1)

df_monthly_counts_all = add_completeness_columns(df_monthly_counts_all)
df_monthly_counts_all = move_column_to_index(df_monthly_counts_all, 'interval_completeness', 1)
  
# We can now remove `num_intervals_expected` from `df_daily_counts` so it's not included in the itable later on
# df_daily_counts = df_daily_counts.drop(columns=['num_intervals_expected'])
df_daily_counts = df_daily_counts.reset_index()


# Devices we couldn't download LE data for (included in devices list but not in LE dict)
devices_with_le_data = list(df_aggregated["device_id"].unique())
devices_without_le_data = [device_id for device_id in devices if device_id not in devices_with_le_data]
df_aggregated_no_data = df_aggregated[df_aggregated['num_intervals'] == 0]
devices_without_le_data = devices_without_le_data + list(df_aggregated_no_data['device_id'].unique())

# Devices with missing LE data
# TODO: add alternative analysis based on timestamp and duration of interval (only works for intervals between existing intervals, i.e. need to handle missing intervals at start or end of period separately)
# Could also just do a quick analysis to verify all intervals have a duration of 300s.
devices_with_missing_le_data = df_aggregated[df_aggregated['num_intervals_missing'] > 0]['device_id'].tolist()

# Devices not meeting data completeness threshold

devices_not_meeting_threshold = df_aggregated[df_aggregated['interval_completeness'] < DATA_COMPLETENESS_THRESHOLD]['device_id'].tolist()

# Devices with complete LE data
devices_with_complete_le_data = df_aggregated[df_aggregated['num_intervals_missing'] == 0]['device_id'].tolist()


# Determine expected number of intervals over the full analysed period
# Calculate the number of 5-minute intervals between the start and end timestamps using pendulum
num_intervals_expected = int((time_end.diff(time_start).in_minutes()) // 5)

# Devices installed within analysed period
devices_installed_in_period = df_aggregated[(df_aggregated['num_intervals_expected'] > 0) & (df_aggregated['num_intervals_expected'] < num_intervals_expected )]['device_id'].tolist() 

# Devices not installed at all
devices_not_installed = df_aggregated[df_aggregated['num_intervals_expected'] == 0]['device_id'].tolist()



In [ ]:
# High level analysis

parameters = [{
  'start_time': time_start,
  'end_time': time_end,
  'num_expected_intervals': num_intervals_expected,
}]

# TODO: update overall completeness calculation as we now take firstLE into account - so different devices have different num_expected_intervals
top_level_stats = [{
  'num_devices': num_devices,
  'overall_completeness': df_aggregated['num_intervals'].sum() / df_aggregated['num_intervals_expected'].sum() * 100,
  'devices_under_threshold': len(devices_not_meeting_threshold),
  'devices_with_missing_intervals': len(devices_with_missing_le_data),
  'devices_without_data': len(devices_without_le_data),
  'devices_with_complete_data': len(devices_with_complete_le_data),
  'devices_installed_during_analysis_period': len(devices_installed_in_period),
  'devices_not_installed': len(devices_not_installed),
  'devices_decommissioned': len(decommissioned_device_info) if 'decommissioned_device_info' in globals() else 0,
}]

df_parameters = pd.DataFrame.from_dict(parameters)
df_stats = pd.DataFrame.from_dict(top_level_stats)

In [ ]:
def trend_graph(df: pd.DataFrame, title_suffix: str = '') -> plotly.graph_objects.Figure:
  # Calculate the average interval_completeness
  average_completeness = df['interval_completeness'].mean()

  # Create the plot
  fig = px.line(
    df, 
    x='date',
    y='interval_completeness', 
    title=f'Interval Completeness Over Time {title_suffix}'
  )
  # Update the x-axis and y-axis labels
  fig.update_layout(
      xaxis_title='Month',
      yaxis_title='Interval Completeness (%)'
  )

  # Add a horizontal line for the average interval_completeness
  fig.add_shape(
      type="line",
      x0=df['date'].min(),
      y0=average_completeness,
      x1=df['date'].max(),
      y1=average_completeness,
      line=dict(
          color="Red",
          width=2,
          dash="dashdot",
      ),
      name="Average Interval Completeness",
      showlegend=True
  )

  # Add the line graph to the legend
  fig.for_each_trace(
      lambda trace: trace.update(name='Interval Completeness'),
  )
  fig.update_traces(showlegend = True)
  return fig

def histogram_across_devices(df: pd.DataFrame, bin_size: int = 5) -> plotly.graph_objects.Figure:
  fig = px.histogram(
    df_aggregated, 
    x="interval_completeness", 
    title="Distribution of interval completeness across devices"
  )

  fig.update_layout(
      xaxis_title='Interval completeness (%)',
      yaxis_title='Number of devices'
  )

  fig.update_traces(xbins=dict( # bins used for histogram
        start=0.0,
        end=100.0,
        size=bin_size
    ))

  return fig



# Outputs

## Aggregate stats

In [ ]:
show(df_parameters)
show(df_stats, 
     columnDefs= [
        { "targets": [1], "createdCell": JavascriptFunction(
                f"""
                    function (td, cellData, rowData, row, col) {{
                        if (cellData < {DATA_COMPLETENESS_THRESHOLD}) {{
                            $(td).css('color', 'red')
                        }}
                    }}
                """
        )},
        {
            "targets": [1],
            "render": JavascriptCode("$.fn.dataTable.render.number(',', '.', 2, '', '%')"),
        }
    ],)



## Decommissioned devices

In [ ]:
decom_display = pd.DataFrame.from_dict(decommissioned_device_info.values()) if decommissioned_device_info else pd.DataFrame(columns=["device_id", "first_heard_at", "last_heard_at"])

if not decom_display.empty:
    decom_display["first_heard_date"] = pd.to_datetime(
        pd.to_numeric(decom_display["first_heard_at"], errors="coerce"), unit="s", utc=True
    ).dt.tz_convert(TIMEZONE).dt.date
    decom_display["last_heard_date"] = pd.to_datetime(
        pd.to_numeric(decom_display["last_heard_at"], errors="coerce"), unit="s", utc=True
    ).dt.tz_convert(TIMEZONE).dt.date
    decom_display["in_analysis_window"] = decom_display["last_heard_at"].apply(
        lambda ts: (
            pendulum.from_timestamp(float(ts), tz=TIMEZONE).end_of("day") >= time_start
            if pd.notna(ts) else False
        )
    )
    in_window = int(decom_display["in_analysis_window"].sum())
    skipped = len(decom_display) - in_window
    logger.info(
        f"{len(decom_display)} decommissioned devices: "
        f"{in_window} included (analysis truncated to last-heard date), "
        f"{skipped} skipped entirely (last heard before {START_DATE})."
    )
    show(
        decom_display[["device_id", "first_heard_date", "last_heard_date", "in_analysis_window"]]
        .sort_values("last_heard_date", ascending=False)
        .reset_index(drop=True),
        showIndex=False,
        buttons=["copyHtml5", "csvHtml5", "excelHtml5"],
    )
else:
    print("No decommissioned devices found.")


## Aggregate per month stats

In [ ]:
show(df_monthly_counts_all, 
     columnDefs= [
        { "targets": [1], "createdCell": JavascriptFunction(
                f"""
                    function (td, cellData, rowData, row, col) {{
                        if (cellData < {DATA_COMPLETENESS_THRESHOLD}) {{
                            $(td).css('color', 'red')
                        }}
                    }}
                """
        )},
        {
            "targets": [1],
            "render": JavascriptCode("$.fn.dataTable.render.number(',', '.', 2, '', '%')"),
        }
    ],
    showIndex=False,
    pageLength=20,
    buttons=["copyHtml5", "csvHtml5", "excelHtml5"],
    maxBytes=0
)

In [ ]:
fig = trend_graph(df_monthly_counts_all)
fig.show()

## Aggregate per day stats

In [ ]:
show(df_daily_counts_all, 
     columnDefs= [
        { "targets": [1], "createdCell": JavascriptFunction(
                f"""
                    function (td, cellData, rowData, row, col) {{
                        if (cellData < {DATA_COMPLETENESS_THRESHOLD}) {{
                            $(td).css('color', 'red')
                        }}
                    }}
                """
        )},
        {
            "targets": [1],
            "render": JavascriptCode("$.fn.dataTable.render.number(',', '.', 2, '', '%')"),
        }
    ],
    showIndex=False,
    pageLength=20,
    buttons=["copyHtml5", "csvHtml5", "excelHtml5"],
    maxBytes=0
)

In [ ]:
fig = trend_graph(df_daily_counts_all)
fig.show()

## Per device stats

In [ ]:
df_aggregated = df_aggregated[['device_id', 'interval_completeness', 'num_intervals_missing']]
show(df_aggregated, 
     columnDefs= [
        { "targets": [1], "createdCell": JavascriptFunction(
                f"""
                    function (td, cellData, rowData, row, col) {{
                        if (cellData < {DATA_COMPLETENESS_THRESHOLD}) {{
                            $(td).css('color', 'red')
                        }}
                    }}
                """
        )},
        {
            "targets": [1],
            "render": JavascriptCode("$.fn.dataTable.render.number(',', '.', 2, '', '%')"),
        }
    ],
    showIndex=False,
    buttons=["copyHtml5", "csvHtml5", "excelHtml5"],
    maxBytes=0
)


In [ ]:
# Visualisation of LE completeness distribution across devices
# Second param is the bin size (i.e. 5 for 20 bins covering 5% each)
fig = histogram_across_devices(df_aggregated, 5)
fig.show()

## Per device per month stats

In [ ]:
show(df_monthly_counts, 
     columnDefs= [
        { "targets": [2], "createdCell": JavascriptFunction(
                f"""
                    function (td, cellData, rowData, row, col) {{
                        if (cellData < {DATA_COMPLETENESS_THRESHOLD}) {{
                            $(td).css('color', 'red')
                        }}
                    }}
                """
        )},
        {
            "targets": [2],
            "render": JavascriptCode("$.fn.dataTable.render.number(',', '.', 2, '', '%')"),
        }
    ],
    showIndex=False,
    pageLength=20,
    buttons=["copyHtml5", "csvHtml5", "excelHtml5"],
    maxBytes=0
)

## Per device per day stats

In [ ]:
show(df_daily_counts, 
     columnDefs= [
        { "targets": [2], "createdCell": JavascriptFunction(
                f"""
                    function (td, cellData, rowData, row, col) {{
                        if (cellData < {DATA_COMPLETENESS_THRESHOLD}) {{
                            $(td).css('color', 'red')
                        }}
                    }}
                """
        )},
        {
            "targets": [2],
            "render": JavascriptCode("$.fn.dataTable.render.number(',', '.', 2, '', '%')"),
        }
    ],
    showIndex=False,
    pageLength=20,
    buttons=["copyHtml5", "csvHtml5", "excelHtml5"],
    maxBytes=0
)

## Analysis of unaggregated data

In [ ]:
if df_unaggregated.empty:
    print("NOTE: Unaggregated interval analysis not configured or too many devices selected. Skipping unaggregated analysis sections.")
else:
    df_unaggregated = df_unaggregated.reset_index(drop=True)


In [ ]:
# Time-of-day across all devices (assumes all devices in same timezone)

In [ ]:
def _resample_dataframe(df: pd.DataFrame, resample_interval: str) -> pd.DataFrame:
    df["datetime"] = (
        pd.to_datetime(df["timestamp"], unit="s").dt.tz_localize("UTC").dt.tz_convert(TIMEZONE)
    )
    df = df.set_index("datetime")

    df_resampled = df.resample(resample_interval).agg(
        duration_sum=("duration", "sum"),
        min_timestamp=("timestamp", "min"),
        max_timestamp=("timestamp", "max"),
        num_intervals=("timestamp", "count"),
    )
    return df_resampled

In [ ]:
def graph_missing_intervals(df: pd.DataFrame, title: str, x: str="datetime", y: str="num_intervals_missing") -> go.Figure:
    df = df.reset_index()
    fig = px.scatter(
        df,
        x=x,
        y=y,
        labels={"num_intervals_missing": "# missing intervals", "datetime": "Time", "time": "Time"},
        title=title
    )
    missing_intervals_max = df["num_intervals_missing"].max()
    range_max = max(missing_intervals_max * 1.1, 1.1)
    range_min = -0.1 if missing_intervals_max < 5 else -1
    dtick = 1 if missing_intervals_max < 10 else 5
    fig.update_yaxes(range=[range_min, range_max], dtick=dtick)
    return fig

In [ ]:
# Time series per device
if df_unaggregated.empty:
    print("Skipping: no unaggregated data available.")
else:
    df_unaggregated_grouped = df_unaggregated.groupby('device_id')
    num_plots: int = len(df_unaggregated_grouped)
    fig = make_subplots(
      rows=num_plots, 
      cols=1,
      subplot_titles=[device_id for device_id in df_unaggregated_grouped.groups.keys()]
    )
    row = 0
    for device_id, df_device in df_unaggregated_grouped:
      row += 1
      print(device_id)
      df_device_resampled = _resample_dataframe(df_device, "5min")
      df_device_resampled["num_intervals_expected"] = 1
      df_device_resampled["num_intervals_missing"] = (df_device_resampled["num_intervals_expected"] - df_device_resampled["num_intervals"]).clip(lower=0)
      device_fig = graph_missing_intervals(df_device_resampled, device_id)
      for trace in device_fig.data:
        fig.add_trace(trace, row=row, col=1)
      fig.update_xaxes(title_text = "Time", row=row, col=1)
      fig.update_yaxes(title_text = "# Intervals missing", 
                       row=row, 
                       col=1, 
                       tickvals= [*range(0, max(2, int(df_device_resampled['num_intervals_missing'].max() + 1)))]
                      )
      
    fig.update_layout(height=num_plots * 270 + 100)
    fig.update_layout(title="Missing Intervals Time Series")
    fig.show()


In [ ]:
# Time-of-day per device
if df_unaggregated.empty:
    print("Skipping: no unaggregated data available.")
else:
    df_unaggregated_grouped = df_unaggregated.groupby('device_id')
    num_plots: int = len(df_unaggregated_grouped)
    fig = make_subplots(
      rows=num_plots, 
      cols=1,
      subplot_titles=[f"{device_id} ({START_DATE} - {END_DATE})" for device_id in df_unaggregated_grouped.groups.keys()],
    )
    row = 0
    for device_id, df_device in df_unaggregated_grouped:
      row += 1
      print(device_id)
      df_device_resampled = _resample_dataframe(df_device, "5min")
      df_device_resampled["num_intervals_expected"] = 1
      df_device_resampled["num_intervals_missing"] = (df_device_resampled["num_intervals_expected"] - df_device_resampled["num_intervals"]).clip(lower=0)
      # Add time-of-day column
      df_device_resampled['time'] = df_device_resampled.index.time
      # Group by time of day and aggregate num_intervals_missing
      df_time_of_day_grouped = df_device_resampled.groupby('time').agg(num_intervals_missing=('num_intervals_missing', 'sum')).reset_index()
      
      device_fig = graph_missing_intervals(df_time_of_day_grouped, f"{device_id} ({START_DATE} - {END_DATE})", "time")
      for trace in device_fig.data:
        fig.add_trace(trace, row=row, col=1)
      fig.update_xaxes(title_text = "Time of day", row=row, col=1)
      fig.update_yaxes(title_text = "# Intervals missing", 
                       row=row, col=1, 
                       tickvals= [*range(0, max(2, int(df_time_of_day_grouped['num_intervals_missing'].max() + 1)))]
                      )

    fig.update_layout(height=num_plots * 270 + 100)
    fig.update_layout(title="Missing Intervals by Time of Day")
    fig.show()
